# Entrega 2: Data Science · Governança, Qualidade e Análise Exploratória
### Projeto Integrador 2º/2026 · Pratô / SAFE · Violência contra a Mulher, aplicativo de apoio camuflado

**Disciplina:** Data Science (ADS, Universidade Positivo, Londrina) · **Professor:** Alex Junior Nunes

**Base:** Atlas da Violência (IPEA e FBSP), séries "Homicídios Mulheres" e "Taxa de Homicídios Mulheres", 27 capitais, 1989 a 2022.

> **Aviso acadêmico:** este notebook é parte de um protótipo acadêmico. Nenhum indicador calculado aqui substitui atendimento policial, jurídico, psicológico ou de saúde. Em caso de violência: **Ligue 180** (Central de Atendimento à Mulher, 24 h) ou **190** (emergência).

**Como executar:** no Google Colab ou no Jupyter, use *Executar tudo*. A primeira célula de carga encontra as bases sozinha: lê da pasta `dados/` no Jupyter local ou baixa do repositório no GitHub quando roda no Colab.

**Mapa do notebook:** 0 Identificação · 1 Governança · 2 Carga e dicionário · 3 Diagnóstico de qualidade · 4 Tratamento · 5 Análise exploratória · 6 Plano das Entregas 3 e 4 · 7 Conclusões e checklist

## Passo 0: Identificação e configuração
Grupo, divisão das etapas do pipeline e bibliotecas usadas no notebook.

In [ ]:
import os
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)
plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.dpi": 100})
COR, COR2, COR3 = "#0F8A54", "#E2674D", "#26343A"   # cores do app: verde profundo, coral e grafite

GRUPO = "Pratô / SAFE"
INTEGRANTES = {   # nome (GitHub) -> etapa do pipeline sob responsabilidade
    "Alana Caled (@alanakaled10)": "Governança, ética e aquisição dos dados",
    "Felipe Trindade (@Trindadefelipe)": "Análise exploratória (EDA)",
    "Higor Bueno (@HigorHBO)": "Qualidade e tratamento dos dados",
    "Higor Domingos (@hdomigos10)": "Modelagem (plano nesta entrega)",
    "João Pedro (@JPmoreiraq1)": "Avaliação (plano nesta entrega)",
}

print("Grupo:", GRUPO)
for nome, etapa in INTEGRANTES.items():
    print(" -", nome, "→", etapa)

## 1. Governança, ética e LGPD
**Responsável:** Alana Caled (@alanakaled10)

### 1.1 Escolha e justificativa da base
A base escolhida é o **Atlas da Violência**, mantido pelo **IPEA** e pelo **Fórum Brasileiro de Segurança Pública (FBSP)**, uma das bases sugeridas no Documento Orientador. Usamos as séries **"Homicídios Mulheres"** e **"Taxa de Homicídios Mulheres"**, no recorte das **27 capitais**.

**Por que esta base serve ao tema e ao aplicativo:** o SAFE é um app de apoio a mulheres em situação de violência. O homicídio de mulheres é a forma mais grave e mais bem registrada dessa violência no Brasil, com série longa (34 anos) e cobertura nacional. A base permite (1) entender onde e quando a violência letal contra a mulher é maior, (2) gerar um **indicador de contexto por capital** que o app pode mostrar junto da Autoavaliação de Risco e do Guia de Direitos, e (3) treinar o pipeline de Data Science sem nenhum dado pessoal.

### 1.2 Ficha da base

| Pergunta | Resposta do grupo |
|---|---|
| Nome da base e link da fonte | Atlas da Violência, tema Violência por Gênero: https://www.ipea.gov.br/atlasviolencia/tema/61/ |
| Órgão responsável e data de acesso | IPEA e FBSP, a partir do Sistema de Informações sobre Mortalidade (SIM) do Ministério da Saúde (MS/SVSA). Download em 18/10/2026 |
| Licença / termos de uso | Dados públicos, divulgados gratuitamente pelo IPEA e pelo FBSP. Uso acadêmico, sempre citando a fonte |
| Granularidade | Cada linha é **uma capital em um ano**: contagem de homicídios de mulheres e taxa por 100 mil mulheres. Não há casos individuais |
| Período e abrangência geográfica | 1989 a 2022 (anual), 27 capitais brasileiras (918 linhas = 27 × 34) |
| Contém dado pessoal ou sensível? | Não. Os dados são **agregados** por capital e ano, sem nome, CPF, endereço, idade ou raça/cor de vítimas |
| Riscos de reidentificação | Baixo. O menor nível é capital por ano. O cuidado existe quando a contagem é muito pequena (0 a 2 mortes no ano em uma capital pequena): somada a notícias locais, poderia apontar um caso. Por isso não publicamos cruzamentos novos com outras bases e não detalhamos casos pequenos |
| Limitações | Ver 1.3 |
| Por que esta base serve ao aplicativo? | Ver 1.1 |

### 1.3 Limitações
- **Subnotificação e classificação da causa da morte:** algumas mortes violentas são registradas como "causa indeterminada" e não entram na contagem de homicídios. O número real tende a ser maior.
- **Mudança de metodologia:** até 1995 os homicídios eram identificados pela CID-9 (códigos E960 a E978); a partir de 1996, pela CID-10. Comparações entre antes e depois de 1996 devem ser feitas com cuidado.
- **Recorte geográfico:** os dados das capitais não representam o interior dos estados.
- **Indisponibilidade:** os recortes por UF e por município apresentaram erro no site ("Nenhum dado disponível para download") na data da coleta. Por isso usamos o recorte por capitais.
- **Tipo de violência:** a base trata só de homicídios. Não inclui agressões, ameaças ou violência psicológica, que são a maioria dos casos atendidos pela rede.

### 1.4 Observações da coleta
- O site informa que os arquivos usam ponto e vírgula, mas os arquivos baixados vieram separados por vírgula.
- A coluna de data vem como "1989-01-15T00:00:00.000Z" e é convertida apenas para o ano.
- As duas séries são unidas em uma tabela pelo código IBGE e pelo ano.
- **Base bruta preservada:** os dois arquivos originais ficam intocados em `data-science/dados/`.

> **LGPD:** trabalhamos só com dados agregados e públicos, nunca tentamos identificar uma pessoa e não coletamos dados reais de vítimas. O aplicativo usa apenas personas fictícias.
>
> **Aviso acadêmico:** qualquer indicador gerado a partir destes dados é uma estimativa acadêmica e não substitui atendimento policial, médico ou jurídico. **Ligue 180** · **190**.

**Autor(a):** Alana Caled (@alanakaled10)

## 2. Carga da base e dicionário de dados
**Responsável:** Alana Caled (@alanakaled10)

### 2.1 Carga dos arquivos originais
A célula abaixo encontra a pasta das bases sozinha: `dados/` no Jupyter local ou o repositório no GitHub quando roda no Colab.

In [ ]:
ARQ_HOMICIDIOS = "Homicídios Mulheres_capitais.csv"
ARQ_TAXA = "Taxa de Homicídios Mulheres_capitais.csv"

if os.path.exists(f"dados/{ARQ_HOMICIDIOS}"):
    PASTA_DADOS = "dados"                      # Jupyter aberto na pasta data-science/
elif os.path.exists(f"data-science/dados/{ARQ_HOMICIDIOS}"):
    PASTA_DADOS = "data-science/dados"         # Jupyter aberto na raiz do repositório
else:
    # Colab: baixa direto do repositório no GitHub, sem precisar enviar arquivos
    PASTA_DADOS = "https://raw.githubusercontent.com/Trindadefelipe/PratoApp/main/data-science/dados"

# Onde salvar os arquivos gerados: na pasta dados/ local ou na pasta atual do Colab
PASTA_SAIDA = "." if PASTA_DADOS.startswith("http") else PASTA_DADOS

def caminho(nome):
    # Nomes com acento e espaço precisam ser codificados quando vêm da internet
    return f"{PASTA_DADOS}/{quote(nome)}" if PASTA_DADOS.startswith("http") else f"{PASTA_DADOS}/{nome}"

print("Lendo as bases de:", PASTA_DADOS)
bruto_hom = pd.read_csv(caminho(ARQ_HOMICIDIOS), sep=",")
bruto_taxa = pd.read_csv(caminho(ARQ_TAXA), sep=",")

print("Homicídios (bruto):", bruto_hom.shape, "| Taxa (bruto):", bruto_taxa.shape)
display(bruto_hom.head())
bruto_hom.info()

### 2.2 Montagem da tabela de trabalho
Cada arquivo tem três colunas: data, código IBGE e valor. Damos nomes simples às colunas, convertemos a data para o ano e unimos as duas séries pelo código IBGE e pelo ano. O nome da capital vem de um mapa feito pela equipe a partir do código IBGE. A tabela `bruto` ainda **não passou por nenhum tratamento**: é a base que o diagnóstico (seção 3) vai medir.

In [ ]:
CAPITAIS = {
    1100205: "Porto Velho", 1200401: "Rio Branco", 1302603: "Manaus",
    1400100: "Boa Vista", 1501402: "Belém", 1600303: "Macapá",
    1721000: "Palmas", 2111300: "São Luís", 2211001: "Teresina",
    2304400: "Fortaleza", 2408102: "Natal", 2507507: "João Pessoa",
    2611606: "Recife", 2704302: "Maceió", 2800308: "Aracaju",
    2927408: "Salvador", 3106200: "Belo Horizonte", 3205309: "Vitória",
    3304557: "Rio de Janeiro", 3550308: "São Paulo", 4106902: "Curitiba",
    4205407: "Florianópolis", 4314902: "Porto Alegre", 5002704: "Campo Grande",
    5103403: "Cuiabá", 5208707: "Goiânia", 5300108: "Brasília",
}

def preparar(tabela, nome_valor):
    t = tabela.copy()
    t.columns = ["periodo", "cod_ibge", nome_valor]           # nomes simples, pela posição
    t["ano"] = pd.to_datetime(t["periodo"]).dt.year          # "1989-01-15T00:00:00.000Z" -> 1989
    return t[["ano", "cod_ibge", nome_valor]]

uniao = preparar(bruto_hom, "homicidios").merge(preparar(bruto_taxa, "taxa"),
                                                on=["cod_ibge", "ano"], how="outer", indicator=True)
cruzamento = uniao["_merge"].value_counts()     # usado no diagnóstico de integridade (3.5)

bruto = uniao.drop(columns="_merge")
bruto["capital"] = bruto["cod_ibge"].map(CAPITAIS)
bruto = bruto[["ano", "cod_ibge", "capital", "homicidios", "taxa"]].sort_values(["cod_ibge", "ano"]).reset_index(drop=True)

print("Tabela de trabalho:", bruto.shape)
print("Anos:", bruto["ano"].min(), "a", bruto["ano"].max(), "| Capitais:", bruto["capital"].nunique())
bruto.to_csv(f"{PASTA_SAIDA}/capitais_homicidios_mulheres.csv", index=False)
print("Base unificada (sem tratamento) salva em:", f"{PASTA_SAIDA}/capitais_homicidios_mulheres.csv")
bruto.head()

### 2.3 Dicionário de dados
Uma linha por coluna: significado, tipo estatístico, unidade e observação. As quatro últimas linhas são colunas **criadas pelo grupo** no tratamento (seção 4).

In [ ]:
dicionario = pd.DataFrame([
    ("ano", "Ano de referência do registro", "quantitativa discreta (tempo)", "ano", "1989 a 2022; vem de uma data ISO convertida"),
    ("cod_ibge", "Código IBGE do município", "qualitativa nominal (identificador)", "—", "7 dígitos; o 1º dígito indica a região"),
    ("capital", "Nome da capital", "qualitativa nominal", "—", "Incluída pela equipe a partir do código IBGE"),
    ("homicidios", "Mulheres vítimas de homicídio no ano", "quantitativa discreta", "óbitos por ano", "Contagem; depende do tamanho da população"),
    ("taxa", "Homicídios de mulheres por 100 mil mulheres", "quantitativa contínua", "óbitos / 100 mil mulheres", "Permite comparar capitais de tamanhos diferentes"),
    ("regiao", "Macrorregião da capital", "qualitativa nominal", "—", "Criada no tratamento (1º dígito do código IBGE)"),
    ("faixa_risco", "Quartil da taxa: Baixo, Médio, Alto, Crítico", "qualitativa ordinal", "—", "Criada no tratamento; quartis da base inteira"),
    ("zero_registrado", "Ano sem homicídio registrado na capital", "qualitativa nominal (binária)", "—", "Criada no tratamento (sinalização)"),
    ("outlier_taxa", "Taxa fora do limite do IQR", "qualitativa nominal (binária)", "—", "Criada no tratamento (sinalização, não exclusão)"),
], columns=["coluna", "significado", "tipo", "unidade", "observação"])
display(dicionario)

**Autor(a):** Alana Caled (@alanakaled10)

## 3. Diagnóstico de qualidade (5 dimensões)
**Responsável:** Higor Bueno (@HigorHBO)

Medimos antes de corrigir, nas cinco dimensões da Aula 9: **completude, validade, consistência, unicidade/integridade e acurácia**.

### 3.1 Completude: quantos valores faltam em cada coluna?

In [ ]:
comp = pd.DataFrame({"nulos": bruto.isna().sum(), "% nulos": (bruto.isna().mean() * 100).round(1)})
display(comp)

ax = comp["% nulos"].plot.barh(color=COR, figsize=(7, 3))
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% de valores vazios")
ax.set_ylabel("coluna")
ax.set_title("Completude por coluna (valores NaN)")
plt.tight_layout()
plt.show()

**Leitura:** nenhuma coluna tem valor vazio: as 918 linhas estão completas. Em uma base agregada isso é comum, mas não basta: a seção seguinte procura buracos que estão "preenchidos".

### 3.2 Buracos disfarçados
Numa base numérica, os buracos disfarçados costumam ser **códigos** (999, 9999, -1) ou **zeros** que significam "sem dado" em vez de "zero de verdade". Contamos os dois.

In [ ]:
CODIGOS_SUSPEITOS = [999, 9999, -1]
suspeitos = {c: int(bruto[c].isin(CODIGOS_SUSPEITOS).sum()) for c in ["homicidios", "taxa"]}
print("Códigos suspeitos (999, 9999, -1):", suspeitos)

zeros = bruto[(bruto["homicidios"] == 0) | (bruto["taxa"] == 0)]
print("Linhas com zero:", len(zeros))
display(zeros)

**Leitura:** não há códigos como 999. Há **6 linhas com zero**: Palmas em 1989, 1990, 1991, 1993 e 1997, e Natal em 2004. Palmas foi criada em 1989 e instalada como capital do Tocantins em 1990, com população pequena nos primeiros anos, então zeros nesse período são **plausíveis como registro real**. Mesmo assim, um zero pode esconder subnotificação. **Decisão:** manter os zeros e sinalizá-los com uma coluna (`zero_registrado`) no tratamento, em vez de apagar ou trocar por outro valor.

### 3.3 Validade: valores fora da faixa possível

In [ ]:
# faixas possíveis (mín, máx): o que ficar fora é INVÁLIDO, não "outlier"
LIMITES = {"ano": (1989, 2022), "homicidios": (0, 5000), "taxa": (0, 100)}

invalidos = {c: int((bruto[c].notna() & ~bruto[c].between(lo, hi)).sum()) for c, (lo, hi) in LIMITES.items()}
invalidos["cod_ibge que não é capital"] = int((~bruto["cod_ibge"].isin(list(CAPITAIS))).sum())
invalidos["homicidios com casa decimal"] = int((bruto["homicidios"] % 1 != 0).sum())
val = pd.Series(invalidos, name="valores inválidos").to_frame()
val["faixa válida"] = [str(LIMITES.get(k, "—")) for k in val.index]
display(val)

**Leitura:** nenhum valor fora da faixa: anos entre 1989 e 2022, contagens inteiras e não negativas, taxas entre 0 e 100, e todos os códigos IBGE correspondem a uma capital.

### 3.4 Consistência: as colunas concordam entre si?
Três verificações: (1) se há 0 homicídios, a taxa também deve ser 0, e vice-versa; (2) cada capital deve ter os 34 anos; (3) a série deve ser comparável no tempo.

In [ ]:
inconsistentes = bruto[(bruto["homicidios"] == 0) != (bruto["taxa"] == 0)]
print("Linhas em que homicídios e taxa discordam sobre o zero:", len(inconsistentes))

anos_por_capital = bruto.groupby("capital")["ano"].nunique()
print("Anos por capital: mínimo", anos_por_capital.min(), "| máximo", anos_por_capital.max())

print("Linhas antes de 1996 (CID-9):", int((bruto["ano"] < 1996).sum()),
      "| a partir de 1996 (CID-10):", int((bruto["ano"] >= 1996).sum()))

**Leitura:** homicídios e taxa concordam em todas as linhas, e todas as capitais têm os 34 anos. A única quebra de consistência é **metodológica**: 189 linhas (1989 a 1995) usam a CID-9 e 729 (1996 a 2022) usam a CID-10. Não dá para "corrigir" isso nos dados; documentamos e levamos em conta ao ler a série no tempo.

### 3.5 Unicidade e integridade

In [ ]:
dup_chave = int(bruto.duplicated(subset=["cod_ibge", "ano"]).sum())
dup_exatas = int(bruto.duplicated().sum())
print("Linhas com a mesma capital e o mesmo ano:", dup_chave)
print("Linhas totalmente idênticas:", dup_exatas)

print("Integridade do cruzamento entre os dois arquivos:")
print(cruzamento.rename({"both": "nas duas séries", "left_only": "só em homicídios", "right_only": "só em taxa"}))

**Leitura:** a chave `[cod_ibge, ano]` é única, e o cruzamento entre os dois arquivos é completo: as 918 linhas existem nas duas séries, sem linha órfã.

**Acurácia** (o dado reflete a realidade?) não pode ser medida só com esta tabela. O grupo verificaria comparando uma amostra de linhas com o painel do Atlas da Violência e com o DataSUS (SIM). Mesmo assim, a acurácia é limitada pela **subnotificação** e pelas mortes registradas como "causa indeterminada" (seção 1.3), que fazem os números ficarem abaixo da realidade.

### 3.6 Resumo do diagnóstico

In [ ]:
resumo = pd.DataFrame([
    ("Completude", "Valores vazios (NaN)", int(bruto.isna().sum().sum()), "Nada a imputar"),
    ("Completude (disfarçada)", "Códigos 999/-1 e zeros", int(sum(suspeitos.values())) + len(zeros), "Zeros mantidos e sinalizados (zero_registrado)"),
    ("Validade", "Valores fora da faixa ou código inválido", int(sum(invalidos.values())), "Regra mantida no tratamento (fora da faixa vira NaN)"),
    ("Consistência", "Homicídios e taxa discordando; anos faltando", len(inconsistentes) + int(34 - anos_por_capital.min()), "Nenhuma correção necessária"),
    ("Consistência (metodologia)", "Troca da CID-9 pela CID-10 em 1996", int((bruto["ano"] < 1996).sum()), "Documentar e ler a série com cuidado"),
    ("Unicidade / Integridade", "Chave repetida; linha sem par entre os arquivos", dup_chave + int(cruzamento.get("left_only", 0) + cruzamento.get("right_only", 0)), "Nenhuma remoção necessária"),
    ("Acurácia", "Subnotificação e causa indeterminada", 0, "Não mensurável aqui; documentar como limitação"),
], columns=["dimensão", "problema verificado", "ocorrências", "decisão"])
display(resumo)

**Autor(a):** Higor Bueno (@HigorHBO)

## 4. Tratamento dos dados
**Responsável:** Higor Bueno (@HigorHBO)

**A ordem importa:** padronizar texto e datas → transformar buracos disfarçados e inválidos em NaN → remover duplicatas → imputar → sinalizar outliers → (só então) escalonar. Trabalhamos em uma **cópia** (`df`) e a base bruta fica preservada.

### 4.1 Padronização de textos, datas e tipos
Os nomes de coluna e as datas já foram padronizados na montagem (2.2). Aqui garantimos os tipos e removemos espaços sobrando nos nomes das capitais.

In [ ]:
df = bruto.copy()
df["capital"] = df["capital"].str.strip()
df = df.astype({"ano": int, "cod_ibge": int, "homicidios": int, "taxa": float})
print(df.dtypes)

### 4.2 Buracos disfarçados e valores inválidos
Valores fora da faixa viram NaN (o diagnóstico mostrou que não há nenhum). Os zeros são mantidos e **sinalizados**.

In [ ]:
n_invalidos = {}
for c, (lo, hi) in LIMITES.items():
    marca = df[c].notna() & ~df[c].between(lo, hi)
    n_invalidos[c] = int(marca.sum())
    df[c] = df[c].mask(marca)
print("Inválidos convertidos em NaN:", n_invalidos)

df["zero_registrado"] = df["homicidios"].eq(0)
print("Linhas sinalizadas como zero_registrado:", int(df["zero_registrado"].sum()))

### 4.3 Duplicatas

In [ ]:
n0 = len(df)
df = df.drop_duplicates().drop_duplicates(subset=["cod_ibge", "ano"], keep="first").reset_index(drop=True)
print(f"{n0} -> {len(df)} linhas ({n0 - len(df)} duplicatas removidas)")

### 4.4 Imputação
Regra da Aula 9: numéricas → **mediana** quando há assimetria (média só se simétrica), com uma coluna-bandeira marcando o que foi imputado. A tabela abaixo mostra a assimetria e quanto falta em cada coluna.

In [ ]:
dist = pd.DataFrame({"média": df[["homicidios", "taxa"]].mean(), "mediana": df[["homicidios", "taxa"]].median(),
                     "assimetria": df[["homicidios", "taxa"]].apply(lambda s: stats.skew(s.dropna())),
                     "% faltando": df[["homicidios", "taxa"]].isna().mean() * 100}).round(2)
display(dist)

faltando = int(df[["homicidios", "taxa"]].isna().sum().sum())
if faltando == 0:
    print("Nenhum valor faltando: não há o que imputar.")
else:
    for c in ["homicidios", "taxa"]:
        df[c + "_imputado"] = df[c].isna()
        df[c] = df[c].fillna(df[c].median())
    print("Valores imputados com a mediana:", faltando)

**Leitura:** não há valor faltando, então nenhuma imputação foi feita. Se houvesse, usaríamos a **mediana**, porque as duas variáveis são assimétricas à direita (a média fica acima da mediana).

### 4.5 Outliers: sinalizar, não apagar (critério de Tukey / IQR)

In [ ]:
q1, q3 = df["taxa"].quantile([.25, .75])
iqr = q3 - q1
lim_inf, lim_sup = max(0.0, q1 - 1.5 * iqr), q3 + 1.5 * iqr
df["outlier_taxa"] = ~df["taxa"].between(lim_inf, lim_sup)

print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f} | limite superior: {lim_sup:.2f}")
print(f"Outliers da taxa: {int(df['outlier_taxa'].sum())} ({df['outlier_taxa'].mean() * 100:.1f}% das linhas)")
display(df.loc[df["outlier_taxa"], ["capital", "ano", "taxa"]].sort_values("taxa", ascending=False))

> ⚠️ **Decisão sobre os outliers:** os valores extremos (16 linhas, com a taxa máxima de 18,59 em Porto Velho, 2001) **não foram excluídos nem imputados**. São registros reais de anos de violência mais alta, não erros de coleta. Apagá-los esconderia justamente os cenários de maior risco que o SAFE quer apoiar. A sensibilidade a esses extremos será tratada na modelagem (escalonador robusto ou faixas de risco).

### 4.6 Novas colunas: macrorregião e faixa de risco
1. `regiao`: macrorregião oficial (Norte, Nordeste, Sudeste, Sul, Centro-Oeste), pelo primeiro dígito do código IBGE.
2. `faixa_risco`: taxa em quatro classes ordinais (Baixo, Médio, Alto, Crítico) pelos quartis da base.

In [ ]:
MAPA_REGIOES = {1: "Norte", 2: "Nordeste", 3: "Sudeste", 4: "Sul", 5: "Centro-Oeste"}
df["regiao"] = df["cod_ibge"].astype(str).str[0].astype(int).map(MAPA_REGIOES)
df["faixa_risco"] = pd.qcut(df["taxa"], q=4, labels=["Baixo", "Médio", "Alto", "Crítico"])

print("Capitais por região:")
print(df.groupby("regiao")["capital"].nunique())
print("Linhas por faixa de risco:")
print(df["faixa_risco"].value_counts(sort=False))

### 4.7 Escalonamento (demonstração para a Entrega 3)
Só uma demonstração, em uma **cópia**: *min-max* e *z-score* na taxa e nos homicídios. Duas lições da Aula 9: (1) escalonar **não muda o formato** da distribuição; (2) na modelagem o escalonador aprende (`fit`) **só no treino**. Por isso as colunas escalonadas **não** vão para a base tratada salva.

In [ ]:
demo = df[["homicidios", "taxa"]].copy()
mm = pd.DataFrame(MinMaxScaler().fit_transform(demo), columns=demo.columns)
zs = pd.DataFrame(StandardScaler().fit_transform(demo), columns=demo.columns)
compara = pd.DataFrame({"assimetria original": demo.apply(stats.skew), "assimetria min-max": mm.apply(stats.skew),
                        "assimetria z-score": zs.apply(stats.skew)}).round(3)
display(compara)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for eixo, dados_hist, cor, titulo in [(ax[0], demo["taxa"], COR, "Taxa original"),
                                      (ax[1], mm["taxa"], COR2, "Taxa min-max [0, 1]"),
                                      (ax[2], zs["taxa"], COR3, "Taxa z-score")]:
    eixo.hist(dados_hist, bins=30, color=cor)
    eixo.set_title(titulo)
    eixo.set_xlabel("valor")
    eixo.set_ylabel("linhas")
plt.tight_layout()
plt.show()

**Leitura:** a assimetria é a mesma nas três versões e os três histogramas têm o mesmo formato; só a escala do eixo muda. Escalonar não "corrige" assimetria nem outliers.

### 4.8 Tabela de decisões e antes × depois

In [ ]:
decisoes = pd.DataFrame([
    ("Nomes de coluna e datas", "Nomes simples; data ISO convertida para o ano", "Padronizar antes de qualquer análise", "2 arquivos com 3 colunas → 1 tabela com 5"),
    ("União das séries", "Juntar homicídios e taxa por código IBGE e ano", "As duas medidas descrevem a mesma capital no mesmo ano", "918 linhas casadas, nenhuma órfã"),
    ("Nome da capital", "Mapa feito pela equipe a partir do código IBGE", "O arquivo só traz o código", "27 capitais identificadas"),
    ("Valores inválidos", "Fora da faixa (LIMITES) → NaN", "Erro de registro não é medida", "0 encontrados"),
    ("Zeros", "Manter e sinalizar (zero_registrado)", "Plausíveis (Palmas nos primeiros anos), mas podem esconder subnotificação", "6 linhas sinalizadas"),
    ("Duplicatas", "Remover chave capital + ano repetida", "Contariam o mesmo ano duas vezes", "0 removidas"),
    ("Imputação", "Nenhuma; regra seria a mediana", "Não há valor faltando", "0 imputados"),
    ("Outliers", "Manter e sinalizar (outlier_taxa, IQR)", "São anos reais de violência mais alta", "16 linhas sinalizadas"),
    ("Novas colunas", "regiao e faixa_risco", "Comparar regiões e preparar uma classe para o modelo", "+2 colunas"),
    ("Escalonamento", "Só demonstrado; não salvo na base", "Na Entrega 3 o fit será só no treino (evita vazamento)", "—"),
], columns=["tema", "o que fizemos", "por quê", "impacto"])
display(decisoes)

antes_depois = pd.DataFrame({
    "bruta": [len(bruto), bruto.shape[1], int(bruto.isna().sum().sum()), int(bruto.duplicated(subset=["cod_ibge", "ano"]).sum()),
              int(sum(invalidos.values())), "—", "—", bruto["taxa"].max()],
    "tratada": [len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated(subset=["cod_ibge", "ano"]).sum()),
                0, int(df["zero_registrado"].sum()), int(df["outlier_taxa"].sum()), df["taxa"].max()],
}, index=["linhas", "colunas", "valores NaN", "chaves repetidas", "valores inválidos", "zeros sinalizados", "outliers sinalizados", "taxa máxima"])
display(antes_depois)

COLUNAS_SAIDA = ["ano", "cod_ibge", "capital", "regiao", "homicidios", "taxa", "faixa_risco", "zero_registrado", "outlier_taxa"]
df[COLUNAS_SAIDA].to_csv(f"{PASTA_SAIDA}/capitais_dados_processados.csv", index=False)
print("Base tratada salva em:", f"{PASTA_SAIDA}/capitais_dados_processados.csv", "| a base bruta continua intocada")

**Autor(a):** Higor Bueno (@HigorHBO)

## 5. Análise exploratória (EDA)
**Responsável:** Felipe Trindade (@Trindadefelipe)

Trabalhamos sobre a base **tratada** (`df`). Cada gráfico tem título, rótulos nos eixos e uma frase de leitura. Estatística descritiva mostra **associação, não causa**, e os dados medem o que **foi registrado**, não tudo o que aconteceu (subnotificação).

### 5.1 Visão geral

In [ ]:
print("Período:", df["ano"].min(), "a", df["ano"].max(), "| Registros:", len(df), "| Capitais:", df["capital"].nunique())
display(df[["homicidios", "taxa"]].describe().round(2))

### 5.2 Variáveis numéricas: tendência central, dispersão e forma

In [ ]:
def resumo_num(s):
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75])
    return {"média": s.mean(), "mediana": s.median(), "moda": s.mode().iloc[0], "desvio-padrão": s.std(),
            "variância": s.var(), "CV (%)": s.std() / s.mean() * 100, "mín": s.min(), "Q1": q1, "Q3": q3,
            "máx": s.max(), "amplitude": s.max() - s.min(), "IQR": q3 - q1,
            "assimetria": stats.skew(s), "curtose": stats.kurtosis(s)}

tab_num = pd.DataFrame({c: resumo_num(df[c]) for c in ["homicidios", "taxa"]}).T.round(2)
display(tab_num)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
rotulos = {"homicidios": "homicídios no ano (óbitos)", "taxa": "taxa por 100 mil mulheres"}
for i, c in enumerate(["homicidios", "taxa"]):
    vals = df[c]
    axes[i, 0].hist(vals, bins=30, color=COR)
    axes[i, 0].axvline(vals.mean(), color=COR2, ls="--", label="média")
    axes[i, 0].axvline(vals.median(), color=COR3, label="mediana")
    axes[i, 0].set_title(f"Distribuição de {c}")
    axes[i, 0].set_xlabel(rotulos[c])
    axes[i, 0].set_ylabel("linhas (capital × ano)")
    axes[i, 0].legend(fontsize=8)
    axes[i, 1].boxplot(vals, flierprops=dict(markersize=3))
    axes[i, 1].set_title(f"Boxplot de {c}")
    axes[i, 1].set_xticks([1])
    axes[i, 1].set_xticklabels([c])
    axes[i, 1].set_ylabel(rotulos[c])
plt.tight_layout()
plt.show()

**Leitura:**
- **Homicídios (contagem):** média de 38,42 contra mediana de 20, moda de 12, desvio-padrão de 55,67 e coeficiente de variação de cerca de 145%. A assimetria é forte (cerca de 4,1) e a curtose alta (cerca de 21): poucas capitais muito populosas, como São Paulo (477 mortes em 1999), puxam a média para cima.
- **Taxa por 100 mil mulheres:** média de 4,19 e mediana de 3,79, muito mais próximas. A assimetria é moderada (cerca de 1,0) e o CV de cerca de 57%. A amplitude vai de 0 a 18,59 (Porto Velho, 2001). A moda é 0 por causa dos 6 anos sem registro (seção 3.2).
- **Consequência:** a **taxa** é a medida certa para comparar capitais, porque não depende do tamanho da população. Para resumir grupos, a **mediana** é mais segura que a média, e na modelagem convém um escalonador robusto.

### 5.3 Distribuição de frequências (absoluta, relativa e acumulada)
Para a taxa (agrupada em classes) e para as variáveis categóricas `regiao` e `faixa_risco`.

In [ ]:
def freq(s, ordenar_por_indice=False):
    f = s.value_counts(dropna=False)
    if ordenar_por_indice:
        f = f.sort_index()
    t = pd.DataFrame({"frequência absoluta": f, "frequência relativa (%)": (f / f.sum() * 100).round(1)})
    t["frequência acumulada (%)"] = t["frequência relativa (%)"].cumsum().round(1)
    return t

df["classe_taxa"] = pd.cut(df["taxa"], bins=[0, 3, 6, 9, 12, 15, 20],
                           labels=["0 a 3", "3 a 6", "6 a 9", "9 a 12", "12 a 15", "15 a 20"], include_lowest=True)

for coluna, ordinal in [("classe_taxa", True), ("regiao", False), ("faixa_risco", True)]:
    print(f"== {coluna} (moda: {df[coluna].mode().iloc[0]})")
    display(freq(df[coluna], ordenar_por_indice=ordinal))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for eixo, coluna, ordinal in zip(axes, ["classe_taxa", "regiao", "faixa_risco"], [True, False, True]):
    contagem = df[coluna].value_counts()
    contagem = contagem.sort_index() if ordinal else contagem.sort_values(ascending=False)
    contagem.plot.bar(ax=eixo, color=COR)
    eixo.set_title(f"Frequência de {coluna}")
    eixo.set_xlabel(coluna)
    eixo.set_ylabel("linhas (capital × ano)")
    eixo.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**Leitura:**
- **Taxa em classes:** 36,2% das linhas ficam entre 0 e 3 e 43,8% entre 3 e 6, ou seja, **80,0% acumulado até 6 por 100 mil**. Acima de 9 há só 32 linhas (3,5%), a cauda à direita vista no histograma.
- **Região:** o Nordeste tem mais linhas (306, 33,3%) só porque tem mais capitais (9). Essa frequência mostra o **tamanho da amostra por região**, não onde há mais violência; para isso usamos a taxa (seção 5.5).
- **Faixa de risco:** cerca de 25% em cada classe, como esperado, porque as faixas foram construídas pelos quartis.

### 5.4 Série no tempo

In [ ]:
por_ano = df.groupby("ano").agg(taxa_media=("taxa", "mean"), homicidios_total=("homicidios", "sum"))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(por_ano.index, por_ano["taxa_media"], color=COR, marker="o", lw=1.8, label="média das 27 capitais")
ax[0].plot(por_ano.index, por_ano["taxa_media"].rolling(3, center=True).mean(), color=COR2, lw=2.5, label="média móvel de 3 anos")
ax[0].axvline(1996, color=COR3, ls="--", lw=1.2, label="1996: troca da CID-9 pela CID-10")
ax[0].set_title("Taxa média de homicídios de mulheres nas capitais")
ax[0].set_xlabel("ano")
ax[0].set_ylabel("taxa média por 100 mil mulheres")
ax[0].legend(fontsize=8)

ax[1].bar(por_ano.index, por_ano["homicidios_total"], color=COR)
ax[1].set_title("Total de homicídios de mulheres nas 27 capitais")
ax[1].set_xlabel("ano")
ax[1].set_ylabel("óbitos no ano")
plt.tight_layout()
plt.show()

periodos = pd.cut(df["ano"], bins=[1988, 1999, 2009, 2019, 2022], labels=["1989-1999", "2000-2009", "2010-2019", "2020-2022"])
display(df.groupby(periodos, observed=True)["taxa"].mean().round(2).to_frame("taxa média"))
print("Ano com a maior taxa média:", por_ano["taxa_media"].idxmax(), "| ano com a menor:", por_ano["taxa_media"].idxmin())

**Leitura:** a taxa média das capitais subiu de 2,58 em 1989 para o pico de 6,21 em 2014 e caiu para 3,42 em 2022. Por período: 2,65 (1989-1999), 4,81 (2000-2009), 5,40 (2010-2019) e 3,74 (2020-2022). O total de mortes segue o mesmo desenho, com pico de 1.263 em 2014. **Cuidado ao interpretar:** a linha de 1996 marca a troca da CID, que muda a forma de contar; e uma alta pode refletir mais mortes **ou** melhor registro, o que a EDA sozinha não separa. Não atribuímos a subida nem a queda a uma causa específica.

### 5.5 Relações entre variáveis

In [ ]:
corr = df[["ano", "homicidios", "taxa"]].corr(method="spearman").round(2)
display(corr)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
im = axes[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xticks(range(len(corr)))
axes[0].set_yticks(range(len(corr)))
axes[0].set_xticklabels(corr.columns)
axes[0].set_yticklabels(corr.index)
axes[0].grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[0].text(j, i, corr.iloc[i, j], ha="center", va="center", fontsize=9)
axes[0].set_title("Correlação de Spearman")
axes[0].set_xlabel("variável")
axes[0].set_ylabel("variável")
fig.colorbar(im, ax=axes[0], shrink=.8)

ordem = df.groupby("regiao")["taxa"].median().sort_values(ascending=False).index
axes[1].boxplot([df.loc[df["regiao"] == r, "taxa"] for r in ordem], showfliers=False)
axes[1].set_xticks(range(1, len(ordem) + 1))
axes[1].set_xticklabels(list(ordem))
axes[1].set_title("Taxa por região (sem outliers no desenho)")
axes[1].set_xlabel("região")
axes[1].set_ylabel("taxa por 100 mil mulheres")

media_capital = df.groupby("capital")["taxa"].mean().sort_values()
axes[2].barh(media_capital.index, media_capital.values, color=COR)
axes[2].set_title("Taxa média por capital (1989-2022)")
axes[2].set_xlabel("taxa média por 100 mil mulheres")
axes[2].set_ylabel("capital")
axes[2].tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

**Leitura:**
- **Correlação (Spearman, robusta à assimetria):** a taxa tem associação positiva moderada com o ano (cerca de 0,38), o que acompanha a subida até 2014 vista na série. Homicídios e taxa têm associação positiva fraca a moderada (cerca de 0,39): capitais grandes têm muitas mortes, mas não necessariamente a maior taxa. Correlação não indica causa.
- **Região:** o Norte tem a maior taxa mediana (cerca de 4,7 por 100 mil) e o Sul a menor (cerca de 2,8).
- **Capitais:** Vitória e Porto Velho têm as maiores médias do período (6,32), seguidas de Boa Vista (5,94); Florianópolis tem a menor (2,35).

### 5.6 Três perguntas respondidas com os dados
Escolhemos perguntas que ajudam o aplicativo: onde a violência letal é maior, como ela mudou no tempo e se o contexto regional difere.

In [ ]:
# Pergunta 1: quais capitais têm as maiores e as menores taxas médias no período?
p1 = df.groupby("capital")["taxa"].mean().sort_values(ascending=False).round(2)
print("P1, 3 maiores:", p1.head(3).to_dict())
print("P1, 3 menores:", p1.tail(3).to_dict())
print("P1, razão entre a maior e a menor:", round(p1.iloc[0] / p1.iloc[-1], 1))

# Pergunta 2: a taxa média das capitais mudou entre os períodos?
p2 = df.groupby(periodos, observed=True)["taxa"].mean().round(2)
print("P2, taxa média por período:", p2.to_dict())

# Pergunta 3: qual região tem a maior taxa típica (mediana, por ser assimétrica)?
p3 = df.groupby("regiao")["taxa"].median().sort_values(ascending=False).round(2)
print("P3, taxa mediana por região:", p3.to_dict())

**Pergunta 1: onde a violência letal contra a mulher é maior?** No período de 1989 a 2022, **Vitória e Porto Velho** têm a maior taxa média (6,32 por 100 mil mulheres), seguidas de **Boa Vista** (5,94). **Florianópolis** tem a menor (2,35): a maior capital tem uma taxa cerca de 2,7 vezes a menor. Para o app, isso indica que o mesmo "nível de atenção" pode significar contextos bem diferentes conforme a cidade.

**Pergunta 2: a violência letal mudou ao longo do tempo?** Sim. A taxa média passou de 2,65 (1989-1999) para 4,81 (2000-2009) e 5,40 (2010-2019), e caiu para 3,74 (2020-2022). A comparação com o período antes de 1996 é limitada pela troca da CID, e a EDA mostra a mudança, mas não explica por que ela aconteceu.

**Pergunta 3: o contexto regional difere?** Sim. O **Norte** tem a maior taxa mediana (4,70 por 100 mil) e o **Sul** a menor (2,82). Centro-Oeste (3,75), Sudeste (3,58) e Nordeste (3,56) ficam no meio. A região é, portanto, uma variável candidata para o modelo da Entrega 3.

**Autor(a):** Felipe Trindade (@Trindadefelipe)

## 6. Plano das Entregas 3 e 4: modelagem e avaliação
Os responsáveis por **Modelagem** e **Avaliação** registram aqui o planejamento, sem treinar nenhum modelo ainda. As decisões respeitam o que o diagnóstico e a EDA mostraram: a base é **agregada** (capital × ano), tem só 918 linhas e é uma **série no tempo**.

A célula abaixo prepara o alvo planejado e calcula os **pisos** que qualquer modelo precisa superar.

In [ ]:
plano = df.sort_values(["cod_ibge", "ano"]).copy()
plano["alvo_alto"] = plano["faixa_risco"].isin(["Alto", "Crítico"]).astype(int)    # 1 = taxa acima da mediana
plano["taxa_ano_anterior"] = plano.groupby("cod_ibge")["taxa"].shift(1)
plano["alvo_ano_anterior"] = plano.groupby("cod_ibge")["alvo_alto"].shift(1)
base_plano = plano.dropna(subset=["taxa_ano_anterior"])

treino = base_plano[base_plano["ano"] <= 2015]
teste = base_plano[base_plano["ano"] >= 2016]
print(f"Linhas com ano anterior: {len(base_plano)} | treino (até 2015): {len(treino)} | teste (2016 em diante): {len(teste)}")
print("Proporção de 'alto' no treino:", round(treino["alvo_alto"].mean(), 3), "| no teste:", round(teste["alvo_alto"].mean(), 3))

piso_majoritaria = max(teste["alvo_alto"].mean(), 1 - teste["alvo_alto"].mean())
piso_persistencia = (teste["alvo_alto"] == teste["alvo_ano_anterior"]).mean()
print(f"Piso 1, sempre chutar a classe majoritária: {piso_majoritaria:.1%}")
print(f"Piso 2, repetir a classe do ano anterior: {piso_persistencia:.1%}  ← o piso que o modelo precisa superar")

### 6.1 Plano de modelagem (Entrega 3)
**Responsável:** Higor Domingos (@hdomigos10)

| Item | Decisão do grupo |
|---|---|
| Objetivo do modelo (o que o app vai mostrar?) | Um **indicador de contexto da capital**: se a taxa de homicídios de mulheres da capital tende a ficar **acima da mediana histórica** no ano seguinte. Na Tela 17 (Resultado da Autoavaliação), ele aparece só como informação complementar ("a sua capital tem taxa acima da média"), com o aviso acadêmico. **Não** é um risco individual da usuária |
| Supervisionado ou não supervisionado? Por quê? | **Supervisionado** (classificação binária) para o indicador, porque a base tem um valor observado para cada capital e ano. **Não supervisionado** (K-Means) como complemento, para agrupar capitais com trajetórias parecidas e descrever perfis |
| Variável-alvo e como foi definida | `alvo_alto` = 1 quando a `faixa_risco` é Alto ou Crítico (taxa acima da mediana). Na Entrega 3, os limites das faixas serão calculados **só com os anos de treino**, para não vazar informação do teste |
| Variáveis candidatas | Taxa do ano anterior, média móvel dos 3 anos anteriores, variação em relação ao ano anterior, `regiao` (codificada) e `zero_registrado`. Para o K-Means: taxa média, tendência e variabilidade de cada capital |
| Variáveis que **não** devem entrar | A **taxa do próprio ano** (é a origem do alvo: vazamento direto), `faixa_risco` do próprio ano, `homicidios` do próprio ano e `cod_ibge`/`capital` como número (identificam a linha, não explicam) |
| Técnicas | **Regressão logística** como base interpretável e **random forest** para comparar; **K-Means** para os perfis de capitais |
| Pré-processamento | Escalonamento (preferência pelo `RobustScaler`, pela assimetria da taxa) e codificação da região **dentro de um Pipeline**, com `fit` só no treino |

**Autor(a):** Higor Domingos (@hdomigos10)

### 6.2 Plano de avaliação (Entregas 3 e 4)
**Responsável:** João Pedro (@JPmoreiraq1)

| Item | Decisão do grupo |
|---|---|
| Métricas e qual priorizar | **Recall da classe "alto"**: deixar de sinalizar uma capital com taxa alta é o pior erro para o app. Junto: precisão, F1 e matriz de confusão. A acurácia sozinha engana, porque as classes estão quase equilibradas (cerca de 50%) |
| Pisos de comparação | O modelo precisa superar os dois pisos da célula acima, principalmente o de **repetir a classe do ano anterior** (cerca de 78% no teste). Se não superar, o indicador simples é melhor que o modelo |
| Validação | Divisão **temporal**: treino até 2015 e teste de 2016 em diante (nunca embaralhar anos). No treino, validação cruzada com `TimeSeriesSplit` de 5 partes |
| Avaliação do K-Means | Coeficiente de silhueta para escolher o número de grupos e leitura dos perfis (taxa média e tendência de cada grupo) |
| Riscos | (1) **Vazamento**: usar a taxa do mesmo ano ou faixas calculadas com a base toda; (2) **amostra pequena**: 27 capitais e poucos anos de teste; (3) **falácia ecológica**: um indicador da capital não diz nada sobre o risco de uma mulher específica; (4) **subnotificação** e troca da CID em 1996; (5) **uso indevido**: uma capital "abaixo da média" não pode passar falsa sensação de segurança |
| Integração ao app (Entrega 4) | Exibir o indicador só como contexto, sempre com o aviso "estimativa acadêmica" e os botões do 180 e do 190 |

**Autor(a):** João Pedro (@JPmoreiraq1)

## 7. Conclusões e checklist
**Responsável:** grupo

**Conclusões:** a base do Atlas da Violência chegou completa: 918 linhas (27 capitais × 34 anos), sem valores vazios, sem códigos inválidos, sem duplicatas e com o cruzamento entre as duas séries perfeito. Os pontos de atenção da qualidade foram os **6 zeros** (Palmas nos primeiros anos e Natal em 2004), mantidos e sinalizados, e a **troca da CID em 1996**, que limita a comparação no tempo. O tratamento não apagou nem inventou dados: só padronizou, sinalizou zeros e 16 outliers e criou região e faixa de risco. A EDA mostrou que a **taxa** é a medida certa para comparar capitais (a contagem é muito assimétrica, puxada por São Paulo), que a taxa média subiu até 2014 e caiu depois, e que o Norte e as capitais Vitória, Porto Velho e Boa Vista têm as taxas mais altas. **Limitações:** dados agregados (sem informação individual), só homicídios, só capitais, subnotificação e associação sem causa.

**Checklist da Entrega 2:**

- [x] Ficha da base preenchida (fonte, licença, LGPD, limitações) e aviso acadêmico presente
- [x] Dicionário de dados completo, com tipo estatístico e unidade
- [x] Diagnóstico nas 5 dimensões, com contagens e exemplos
- [x] Tratamento documentado (tabela de decisões e antes × depois), base bruta preservada e base tratada salva em `data-science/dados/`
- [x] EDA com tendência central, dispersão, forma, frequências, série no tempo e relações, cada gráfico com título, eixos e frase de leitura
- [x] 3 perguntas respondidas com os dados, sem concluir causa a partir de correlação
- [x] Plano da modelagem e da avaliação (alvo, variáveis, métricas, validação, riscos)
- [x] Cada seção assinada (**Autor(a)**)
- [ ] Commits individuais de todos os integrantes de Data Science
- [ ] `USO_IA.md` atualizado por todos
- [ ] Notebook salvo **com as saídas visíveis** (Executar tudo)

> Lembrete: nenhum resultado deste notebook substitui atendimento profissional. **Ligue 180** (24 h) · **190** (emergência).